# PatchCore — Analisi Risultati
Notebook per la visualizzazione dei risultati prodotti da `evaluate()` e `evaluate_from_db()`.

**Input supportati:**
- CSV da `evaluate()` — colonne: `name, mu, sigma, threshold, free_scores, fp_count, obstr_score, is_tp`
- CSV da `evaluate_from_db()` — colonne: `reference_id, test_id, test_type, venue_type, file_path, score, threshold, is_anomaly`
- SQLite DB (opzionale) — tabelle `experiments` + `results`

Imposta i path nella cella di configurazione, poi esegui tutto con **Run All**.

In [ ]:
# ── Dipendenze ────────────────────────────────────────────────────────────────
# pip install pandas matplotlib seaborn scikit-learn opencv-python pillow

import json
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
print('Librerie caricate.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURAZIONE — modifica questi path
# ══════════════════════════════════════════════════════════════════════════════

# Scegli la sorgente dati: 'evaluate' oppure 'evaluate_db'
SOURCE = 'evaluate_db'   # <── cambia qui

# Path CSV (prodotto da evaluate() o evaluate_from_db())
CSV_PATH = 'eval_results.csv'

# Path SQLite DB (solo se SOURCE == 'evaluate_db')
DB_PATH = '../Aggregated_dataset_db/occlusion.db'

# Directory con le immagini originali (per visualizzare heatmap salvate)
# Struttura attesa: OUTPUT_DIR/<stem>_heatmap.jpg  e  <stem>_overlay.jpg
OUTPUT_DIR = 'patchcore_output'   # prodotto da test_image()

# Cartella in cui salvare i grafici generati da questo notebook
PLOTS_DIR = 'plots'
Path(PLOTS_DIR).mkdir(exist_ok=True)

print(f'SOURCE      = {SOURCE}')
print(f'CSV_PATH    = {CSV_PATH}')
print(f'PLOTS_DIR   = {PLOTS_DIR}')

---
## 1. Caricamento dati

In [ ]:
# ── Carica CSV e normalizza in un formato comune ──────────────────────────────

def load_evaluate_csv(path: str) -> pd.DataFrame:
    """CSV da evaluate() — una riga per immagine di background."""
    df = pd.read_csv(path)
    # free_scores è salvato come stringa JSON '[0.123, ...]'
    df['free_scores'] = df['free_scores'].apply(json.loads)
    # Espandi in righe separate (una per ogni test free)
    free_rows = []
    for _, row in df.iterrows():
        for s in row['free_scores']:
            free_rows.append({
                'name': row['name'],
                'score': s,
                'threshold': row['threshold'],
                'test_type': 'normal',
                'is_anomaly': int(s > row['threshold']),
                'venue_type': 'n/a',
            })
        if pd.notna(row.get('obstr_score')):
            free_rows.append({
                'name': row['name'],
                'score': row['obstr_score'],
                'threshold': row['threshold'],
                'test_type': 'obstructed',
                'is_anomaly': int(row['is_tp']) if pd.notna(row.get('is_tp')) else None,
                'venue_type': 'n/a',
            })
    return pd.DataFrame(free_rows)


def load_evaluate_db_csv(path: str) -> pd.DataFrame:
    """CSV da evaluate_from_db() — una riga per coppia (reference, test)."""
    df = pd.read_csv(path)
    df['is_anomaly'] = df['is_anomaly'].astype(int)
    return df


if SOURCE == 'evaluate':
    df = load_evaluate_csv(CSV_PATH)
else:
    df = load_evaluate_db_csv(CSV_PATH)

print(f'Righe caricate: {len(df)}')
print(f'Colonne: {list(df.columns)}')
df.head()

In [ ]:
# ── Split normal / obstructed ─────────────────────────────────────────────────
df_normal = df[df['test_type'] == 'normal'].copy()
df_obstr  = df[df['test_type'] == 'obstructed'].copy()

print(f'Campioni normali   : {len(df_normal)}')
print(f'Campioni ostruiti  : {len(df_obstr)}')

if len(df_obstr) == 0:
    print('⚠  Nessun campione ostruito: i grafici TP/ROC non saranno disponibili.')

---
## 2. Distribuzione degli anomaly score

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Istogramma sovrapposto ─────────────────────────────────────────────────────
ax = axes[0]
bins = np.linspace(
    df['score'].min() * 0.95,
    df['score'].max() * 1.05,
    50
)
ax.hist(df_normal['score'],  bins=bins, alpha=0.65, label='Normale',  color='steelblue', edgecolor='white')
if len(df_obstr):
    ax.hist(df_obstr['score'], bins=bins, alpha=0.65, label='Ostruita', color='tomato',    edgecolor='white')

# Soglia media
thr_mean = df['threshold'].mean()
ax.axvline(thr_mean, color='black', linestyle='--', linewidth=1.5, label=f'Soglia media ({thr_mean:.3f})')

ax.set_xlabel('Anomaly score')
ax.set_ylabel('Conteggio')
ax.set_title('Distribuzione anomaly score')
ax.legend()

# ── Box plot per tipo ──────────────────────────────────────────────────────────
ax2 = axes[1]
plot_df = df[['score', 'test_type']].copy()
plot_df['test_type'] = plot_df['test_type'].map({'normal': 'Normale', 'obstructed': 'Ostruita'})
sns.boxplot(data=plot_df, x='test_type', y='score', palette={'Normale': 'steelblue', 'Ostruita': 'tomato'}, ax=ax2)
ax2.axhline(thr_mean, color='black', linestyle='--', linewidth=1.5, label=f'Soglia media')
ax2.set_xlabel('')
ax2.set_ylabel('Anomaly score')
ax2.set_title('Box plot score per classe')
ax2.legend()

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/01_score_distribution.png', bbox_inches='tight')
plt.show()
print('Salvato: 01_score_distribution.png')

---
## 3. Curva ROC e Precision-Recall

In [ ]:
if len(df_obstr) == 0:
    print('Nessun campione ostruito: celle ROC saltate.')
else:
    all_scores = list(df_normal['score']) + list(df_obstr['score'])
    all_labels = [0] * len(df_normal) + [1] * len(df_obstr)

    fpr_arr, tpr_arr, thr_arr = roc_curve(all_labels, all_scores)
    auroc = roc_auc_score(all_labels, all_scores)

    prec_arr, rec_arr, _ = precision_recall_curve(all_labels, all_scores)
    ap = np.trapz(prec_arr[::-1], rec_arr[::-1])  # area under PR curve

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── ROC ───────────────────────────────────────────────────────────────────
    ax = axes[0]
    ax.plot(fpr_arr, tpr_arr, color='darkorange', lw=2, label=f'ROC (AUROC = {auroc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)

    # Punto operativo alla soglia media
    op_idx = np.argmin(np.abs(thr_arr - thr_mean))
    ax.scatter(fpr_arr[op_idx], tpr_arr[op_idx], s=100, zorder=5, color='red',
               label=f'Operating point (thr={thr_mean:.3f})')

    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('Curva ROC')
    ax.legend(loc='lower right')

    # ── Precision-Recall ──────────────────────────────────────────────────────
    ax2 = axes[1]
    ax2.plot(rec_arr, prec_arr, color='steelblue', lw=2, label=f'PR curve (AP = {ap:.3f})')
    ax2.set_xlabel('Recall')
    ax2.set_ylabel('Precision')
    ax2.set_title('Curva Precision-Recall')
    ax2.legend()

    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/02_roc_pr.png', bbox_inches='tight')
    plt.show()
    print(f'AUROC = {auroc:.4f}   AP = {ap:.4f}')
    print('Salvato: 02_roc_pr.png')

---
## 4. FPR / TPR per venue_type

In [ ]:
if 'venue_type' not in df.columns or df['venue_type'].nunique() <= 1:
    print('venue_type non disponibile o unico: cella saltata.')
else:
    venue_stats = []
    for venue, grp in df.groupby('venue_type'):
        norm = grp[grp['test_type'] == 'normal']
        obst = grp[grp['test_type'] == 'obstructed']
        fpr_v = norm['is_anomaly'].mean() if len(norm) else float('nan')
        tpr_v = obst['is_anomaly'].mean() if len(obst) else float('nan')
        venue_stats.append({'venue': venue, 'FPR': fpr_v, 'TPR': tpr_v,
                            'n_normal': len(norm), 'n_obstr': len(obst)})

    vs = pd.DataFrame(venue_stats).set_index('venue')
    display(vs.style.format({'FPR': '{:.2%}', 'TPR': '{:.2%}'}))

    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(len(vs))
    w = 0.35
    ax.bar(x - w/2, vs['FPR'], w, label='FPR (↓ meglio)', color='tomato',    alpha=0.8)
    ax.bar(x + w/2, vs['TPR'], w, label='TPR (↑ meglio)', color='steelblue', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(vs.index, rotation=20)
    ax.set_ylabel('Rate')
    ax.set_title('FPR / TPR per venue type')
    ax.set_ylim(0, 1.1)
    ax.axhline(0.05, color='red',   linestyle=':', lw=1, label='FPR target 5%')
    ax.axhline(0.90, color='green', linestyle=':', lw=1, label='TPR target 90%')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/03_fpr_tpr_per_venue.png', bbox_inches='tight')
    plt.show()
    print('Salvato: 03_fpr_tpr_per_venue.png')

---
## 5. Score per immagine — ranking

In [ ]:
# ── Top-N falsi positivi (normale con score più alto) ─────────────────────────
TOP_N = 15

top_fp = (
    df_normal
    .sort_values('score', ascending=False)
    .head(TOP_N)
    [['name' if 'name' in df.columns else 'file_path', 'score', 'threshold', 'is_anomaly']]
)
top_fp.columns = ['id', 'score', 'threshold', 'is_anomaly']

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['tomato' if a else 'steelblue' for a in top_fp['is_anomaly']]
ax.barh(range(len(top_fp)), top_fp['score'], color=colors)
ax.axvline(top_fp['threshold'].mean(), color='black', linestyle='--', lw=1.5, label='Soglia')
ax.set_yticks(range(len(top_fp)))
ax.set_yticklabels(top_fp['id'], fontsize=8)
ax.set_xlabel('Anomaly score')
ax.set_title(f'Top-{TOP_N} immagini normali per score (rosso = falso positivo)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/04_top_false_positives.png', bbox_inches='tight')
plt.show()
print('Salvato: 04_top_false_positives.png')

In [ ]:
# ── Top-N falsi negativi (ostruita con score più basso) ───────────────────────
if len(df_obstr):
    top_fn = (
        df_obstr
        .sort_values('score', ascending=True)
        .head(TOP_N)
        [['name' if 'name' in df.columns else 'file_path', 'score', 'threshold', 'is_anomaly']]
    )
    top_fn.columns = ['id', 'score', 'threshold', 'is_anomaly']

    fig, ax = plt.subplots(figsize=(12, 4))
    colors = ['steelblue' if a else 'tomato' for a in top_fn['is_anomaly']]
    ax.barh(range(len(top_fn)), top_fn['score'], color=colors)
    ax.axvline(top_fn['threshold'].mean(), color='black', linestyle='--', lw=1.5, label='Soglia')
    ax.set_yticks(range(len(top_fn)))
    ax.set_yticklabels(top_fn['id'], fontsize=8)
    ax.set_xlabel('Anomaly score')
    ax.set_title(f'Top-{TOP_N} immagini ostruite con score più basso (rosso = falso negativo)')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/05_top_false_negatives.png', bbox_inches='tight')
    plt.show()
    print('Salvato: 05_top_false_negatives.png')

---
## 6. Analisi soglia — FPR / TPR vs k·σ

In [ ]:
if len(df_obstr) == 0:
    print('Nessun campione ostruito: cella saltata.')
else:
    # Ricalcola FPR/TPR al variare del moltiplicatore k
    # usando mu e sigma di ogni immagine (già nel CSV per evaluate()) o la
    # soglia fissa per evaluate_from_db() — in quel caso usiamo un range di offset

    k_values = np.arange(0.5, 5.5, 0.25)
    fpr_list, tpr_list = [], []

    if 'mu' in df.columns and 'sigma' in df.columns:
        # evaluate(): ricalcolo esatto
        raw_csv = pd.read_csv(CSV_PATH)
        raw_csv['free_scores'] = raw_csv['free_scores'].apply(json.loads)

        for k in k_values:
            fp_total, fn_total = 0, 0
            free_total, obstr_total = 0, 0
            for _, row in raw_csv.iterrows():
                thr_k = row['mu'] + k * row['sigma']
                for s in row['free_scores']:
                    fp_total   += int(s > thr_k)
                    free_total += 1
                if pd.notna(row.get('obstr_score')):
                    fn_total   += int(row['obstr_score'] <= thr_k)
                    obstr_total += 1
            fpr_list.append(fp_total / free_total  if free_total  else np.nan)
            tpr_list.append(1 - fn_total / obstr_total if obstr_total else np.nan)
    else:
        # evaluate_from_db(): uso percentile della distribuzione score normali come proxy
        mu_global    = df_normal['score'].mean()
        sigma_global = df_normal['score'].std()
        for k in k_values:
            thr_k = mu_global + k * sigma_global
            fpr_list.append((df_normal['score'] > thr_k).mean())
            tpr_list.append((df_obstr['score']  > thr_k).mean())

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(k_values, fpr_list, 'o-', color='tomato',    label='FPR')
    ax.plot(k_values, tpr_list, 's-', color='steelblue', label='TPR')
    ax.axvline(3.0, color='gray', linestyle='--', lw=1.2, label='k=3 (default)')
    ax.set_xlabel('k  (soglia = μ + k·σ)')
    ax.set_ylabel('Rate')
    ax.set_title('Trade-off FPR / TPR al variare di k')
    ax.legend()
    ax.set_ylim(-0.05, 1.1)
    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/06_threshold_sweep.png', bbox_inches='tight')
    plt.show()
    print('Salvato: 06_threshold_sweep.png')

---
## 7. Visualizzazione heatmap (overlay da disco)

In [ ]:
from PIL import Image
import os

def show_heatmaps(output_dir: str, n: int = 6):
    """Mostra i primi N overlay salvati da test_image()."""
    overlays = sorted(Path(output_dir).glob('*_overlay.jpg'))[:n]
    if not overlays:
        print(f'Nessun overlay trovato in {output_dir}. '
              f'Esegui prima test_image() con la flag --img.')
        return

    cols = 3
    rows = int(np.ceil(len(overlays) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 4))
    axes = np.array(axes).flatten()

    for i, p in enumerate(overlays):
        img = Image.open(p)
        axes[i].imshow(img)
        axes[i].set_title(p.stem.replace('_overlay', ''), fontsize=9)
        axes[i].axis('off')

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.suptitle('Overlay heatmap (anomaly score sovrapposto all\'immagine originale)', y=1.01)
    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/07_heatmap_gallery.png', bbox_inches='tight')
    plt.show()
    print('Salvato: 07_heatmap_gallery.png')


show_heatmaps(OUTPUT_DIR, n=9)

---
## 8. Confronto coppia: normale vs ostruita

In [ ]:
def show_pair(normal_overlay_path: str, obstr_overlay_path: str,
              normal_score: float = None, obstr_score: float = None,
              threshold: float = None):
    """
    Mostra side-by-side l'overlay di una scena normale e della sua versione ostruita.

    Uso:
        show_pair(
            'patchcore_output/img001_overlay.jpg',
            'patchcore_output/img001_obstr_overlay.jpg',
            normal_score=0.312, obstr_score=0.871, threshold=0.650
        )
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for ax, path, label, score in [
        (axes[0], normal_overlay_path, 'Normale',  normal_score),
        (axes[1], obstr_overlay_path,  'Ostruita', obstr_score),
    ]:
        if Path(path).exists():
            ax.imshow(Image.open(path))
        else:
            ax.text(0.5, 0.5, f'File non trovato:\n{path}',
                    ha='center', va='center', transform=ax.transAxes, color='red')

        title = label
        if score is not None:
            anomaly = threshold is not None and score > threshold
            status  = '  ANOMALIA' if anomaly else '✅ OK'
            title   = f'{label}  |  score={score:.3f}  {status}'
            color   = 'red' if anomaly else 'green'
            ax.set_title(title, color=color, fontsize=10)
        else:
            ax.set_title(title)
        ax.axis('off')

    if threshold is not None:
        plt.suptitle(f'Soglia: {threshold:.3f}', fontsize=10, color='gray')

    plt.tight_layout()
    plt.show()


# ── Esempio: modifica i path con i tuoi file reali ────────────────────────────
# show_pair(
#     normal_overlay_path = 'patchcore_output/img001_overlay.jpg',
#     obstr_overlay_path  = 'patchcore_output/img001_obstr_overlay.jpg',
#     normal_score = 0.31,
#     obstr_score  = 0.87,
#     threshold    = 0.65,
# )
print('Funzione show_pair() pronta. Decommenta l\'esempio sopra con i tuoi path.')

---
## 9. Riepilogo numerico finale

In [ ]:
print('=' * 55)
print('RIEPILOGO')
print('=' * 55)
print(f'  Campioni normali   : {len(df_normal)}')
print(f'  Campioni ostruiti  : {len(df_obstr)}')
print()
print(f'  Score normali   — μ={df_normal["score"].mean():.4f}  '
      f'σ={df_normal["score"].std():.4f}  '
      f'p95={df_normal["score"].quantile(0.95):.4f}')
if len(df_obstr):
    print(f'  Score ostruiti  — μ={df_obstr["score"].mean():.4f}  '
          f'σ={df_obstr["score"].std():.4f}  '
          f'p5={df_obstr["score"].quantile(0.05):.4f}')
print()
print(f'  Soglia media       : {df["threshold"].mean():.4f}')
print(f'  FPR                : {df_normal["is_anomaly"].mean():.2%}')
if len(df_obstr):
    print(f'  TPR (Recall)       : {df_obstr["is_anomaly"].mean():.2%}')
    print(f'  AUROC              : {roc_auc_score([0]*len(df_normal)+[1]*len(df_obstr), list(df_normal["score"])+list(df_obstr["score"])):.4f}')
print('=' * 55)